# 05 Data-quality checks: Silver
Fails the workflow if Silver has nulls in key columns, duplicate keys, or fewer rows than there are distinct customers in Bronze.

In [ ]:
from pyspark.sql import functions as F
from sales_lakehouse_pipeline import quality

silver = spark.table("silver.sales")
bronze_keys = spark.table("bronze.sales").select("customer_id").distinct().count()

quality.enforce([
    quality.check_row_count(silver, "silver.sales", min_rows=1),
    quality.check_no_nulls(silver, "silver.sales", ["customer_id", "amount", "updated_at"]),
    quality.check_unique_key(silver, "silver.sales", ["customer_id"]),
    quality.check_row_count_not_below(silver, "silver.sales", bronze_keys, "distinct bronze customers"),
])